### Database Query Engine START

In [1]:
# Import the required packages
import os
from os.path import dirname as one_folder_up, join as join_paths
import sqlite3

# Import pandas to read query outputs as dfs, read source files, numpy 
import pandas as pd
import numpy as np

from datetime import datetime

import traceback

C:\Users\tatil\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\tatil\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [2]:
# Define the functions

# START - DB
# Connects to an sqlite db and returns the conn object
def connect_to_database(db_path):
    try:
        conn = sqlite3.connect(db_path)
        return conn
    except Exception as e:
        return False

# Execute a select query against a DB
def execute_db_select(conn, query):
    try:
        df = pd.read_sql_query(query, conn)
        return df
    except Exception as e:
        print(traceback.format_exc())
        return False

# Gets an insert query along with a pandas dataframe, inserts the dataframe into the DB  
def execute_db_insert(conn, query, data):
    try:
        # Create cursor, execute the query, commit, then close the cursor
        cur = conn.cursor()
        cur.executemany(query, data)
        conn.commit()
        cur.close()
        return True
    except Exception as e:
        return False

# Deletes rows from a table based on an array of ids for a particular column
def execute_db_delete(conn, query, ids):
    try:
        # Create cursor, execute the query, commit, then close the cursor
        cur = conn.cursor()
        cur.executemany(query, ids)
        conn.commit()
        cur.close()
        return True
    except Exception as e:
        return False

def execute_db_query(conn, query):
    try:
        # Create cursor, execute the query, commit, then close the cursor
        cur = conn.cursor()
        cur.execute(query)
        conn.commit()
        cur.close()
        return True
    except Exception as e:
        return False
# END - DB

In [3]:
# Define parameters
target_db               = "courses.db"
target_table            = "enrollments"

In [4]:
# Derive required variable(s)
current_folder          = os.getcwd()
root_folder             = one_folder_up(current_folder) 
""" 
    As the current file sits here "\gic_assignment\1_Data_Ingestion_Pipeline\"
    and the root folder is gic_assignment. 
"""

data_folder             = join_paths(root_folder, "data")
output_folder           = join_paths(data_folder, "output")

target_db_path          = join_paths(output_folder, target_db)

### Database Query Engine END

## SQL REPORTING Statement

### Show the current state of the table

In [5]:
conn = connect_to_database(target_db_path)
results = execute_db_select(conn, "SELECT * FROM enrollments")
print(results)
conn.close()

    enrollment_id participant_id participant_name  course_id course_date  \
0               1           P001        Alice Tan          1  2024-01-15   
1               2           P002          Bob Lee          1  2024-01-15   
2               3           P007        Grace Koh          1  2024-01-20   
3               4           P009       Irene Chua          1  2024-01-25   
4               5           P003       Charlie Ng          2  2024-02-01   
5               6           P001        Alice Tan          2  2024-02-01   
6               7           P004        Diana Lim          1  2024-02-10   
7               8           P007        Grace Koh          2  2024-02-15   
8               9           P009       Irene Chua          2  2024-02-20   
9              10           P002          Bob Lee          3  2024-03-05   
10             11           P005       Edward Goh          4  2024-03-15   
11             12           P003       Charlie Ng          1  2024-03-20   
12          

#### Assumptions
- As we only have one date column available in the table, i.e.,. course_date, we have to assume that this is the date on which the participant joined the course. We can also infer this by looking at the course_dates for the course_id 1 when Alice Tan (2024-01-15) and Grace Koh (2024-01-20) enrolled
- As we don't have a course completion date, we have to assume that the enrollee will be able to complete a given course in a week. This will not be the case in real life, but for the purposes of this exercise, we can assume that this is the case

In [37]:
conn = connect_to_database(target_db_path)
reporting_query = """
    WITH RECURSIVE courses_with_prerequisites AS (
        SELECT course_id,
            SUBSTR(prerequisites, 1, INSTR(prerequisites, ',') - 1) AS prerequisite,
            SUBSTR(prerequisites, INSTR(prerequisites, ',') + 1) AS remaining_prerequisites
        FROM
        (
            SELECT course_id,
                REPLACE(REPLACE(REPLACE(prerequisites, '[', ''), ']', ''), ' ', '')  || ',' AS prerequisites
            FROM courses
            WHERE prerequisites IS NOT NULL
        ) AS courses_base

        UNION ALL

        SELECT course_id,
            SUBSTR(remaining_prerequisites, 1, INSTR(remaining_prerequisites, ',') - 1) AS prerequisite,
            SUBSTR(remaining_prerequisites, INSTR(remaining_prerequisites, ',') + 1) AS remaining_prerequisites
        FROM courses_with_prerequisites
        WHERE remaining_prerequisites != ''
    ),

    courses_with_prerequisites_no_nulls AS (
        SELECT course_id,
            prerequisite
        FROM courses_with_prerequisites
        WHERE prerequisite != ''
        AND prerequisite IS NOT NULL
        GROUP BY course_id,
            prerequisite
    ),

    courses_with_prerequisite_counts AS (
        SELECT base.course_id,
            base.prerequisite,
            counts.total_prerequisites
        FROM courses_with_prerequisites_no_nulls base
        INNER JOIN
        (
            SELECT course_id,
                COUNT(prerequisite) AS total_prerequisites
            FROM courses_with_prerequisites_no_nulls
            GROUP BY course_id
        ) counts
        ON base.course_id = counts.course_id
    ),

    enrollments_base AS (
        SELECT participant_id,
            participant_name,
            course_id,
            course_date
        FROM enrollments
    ),

    enrollments_courses AS (
        SELECT E.participant_id,
            E.participant_name,
            E.course_id,
            E.course_date,
            C.total_prerequisites,
            C.prerequisite AS course_id_prerequisite
        FROM enrollments_base E
        LEFT JOIN courses_with_prerequisite_counts C
        ON E.course_id = C.course_id
    )

    SELECT *
    FROM enrollments_courses
    ;
"""
results = execute_db_select(conn, reporting_query)
print(results)
conn.close()

   participant_id participant_name  course_id course_date  \
0            P001        Alice Tan          1  2024-01-15   
1            P002          Bob Lee          1  2024-01-15   
2            P007        Grace Koh          1  2024-01-20   
3            P009       Irene Chua          1  2024-01-25   
4            P003       Charlie Ng          2  2024-02-01   
5            P001        Alice Tan          2  2024-02-01   
6            P004        Diana Lim          1  2024-02-10   
7            P007        Grace Koh          2  2024-02-15   
8            P009       Irene Chua          2  2024-02-20   
9            P002          Bob Lee          3  2024-03-05   
10           P002          Bob Lee          3  2024-03-05   
11           P005       Edward Goh          4  2024-03-15   
12           P003       Charlie Ng          1  2024-03-20   
13           P008        Henry Teo          3  2024-03-25   
14           P008        Henry Teo          3  2024-03-25   
15           P007       

In [5]:
conn = connect_to_database(target_db_path)
results = execute_db_select(conn, "SELECT * FROM courses WHERE prerequisites IS NOT NULL")
print(results)
conn.close()

   course_id              name  \
0          3   CS200 Databases   
1          4  CS201 Networking   
2          5     CS300 Systems   

                                         description prerequisites  
0  Covers relational database design, SQL, normal...         [1,2]  
1  Explores network protocols, TCP/IP stack, rout...           [3]  
2  Advanced course on operating systems, concurre...           [4]  


In [6]:
conn = connect_to_database(target_db_path)
results = execute_db_select(conn, "SELECT strftime('%s', course_date) - strftime('%s', CURRENT_TIMESTAMP) AS COL2 FROM enrollments")
print(results)
conn.close()

        COL2
0  -88849251
1  -88849251
2  -88417251
3  -87985251
4  -87380451
5  -87380451
6  -86602851
7  -86170851
8  -85738851
9  -84529251
10 -83665251
11 -83233251
12 -82801251
13 -82196451
14 -82196451
15 -81850851
16 -81418851
17 -80986851
18 -80382051
19 -79604451
20 -78394851
21 -77962851
22 -77530851
23 -76926051
24 -75716451
25 -75284451
26 -74334051
27 -74334051
28 -73988451
29 -73556451
30 -73124451
31 -72692451


In [11]:
conn = connect_to_database(target_db_path)
results = execute_db_select(conn, """SELECT course_id,
            SUBSTR(REPLACE(REPLACE(REPLACE(prerequisites || ',', '[', ''), ']', ''), ' ', ''), 1, INSTR(REPLACE(REPLACE(REPLACE(prerequisites || ',', '[', ''), ']', ''), ' ', ''), ',') - 1) AS prerequisite,
            SUBSTR(REPLACE(REPLACE(REPLACE(prerequisites || ',', '[', ''), ']', ''), ' ', ''), INSTR(REPLACE(REPLACE(REPLACE(prerequisites || ',', '[', ''), ']', ''), ' ', ''), ',') + 1) AS remaining_prerequisites
        FROM courses
        WHERE prerequisites IS NOT NULL""")
print(results)
conn.close()

   course_id prerequisite remaining_prerequisites
0          3            1                      2,
1          4            3                        
2          5            4                        


In [51]:
conn = connect_to_database(target_db_path)
reporting_query = """
    WITH RECURSIVE courses_with_prerequisites AS (
        SELECT course_id,
            SUBSTR(prerequisites, 1, INSTR(prerequisites, ',') - 1) AS prerequisite,
            SUBSTR(prerequisites, INSTR(prerequisites, ',') + 1) AS remaining_prerequisites
        FROM
        (
            SELECT course_id,
                REPLACE(REPLACE(REPLACE(prerequisites, '[', ''), ']', ''), ' ', '')  || ',' AS prerequisites
            FROM courses
            WHERE prerequisites IS NOT NULL
        ) AS courses_base

        UNION ALL

        SELECT course_id,
            SUBSTR(remaining_prerequisites, 1, INSTR(remaining_prerequisites, ',') - 1) AS prerequisite,
            SUBSTR(remaining_prerequisites, INSTR(remaining_prerequisites, ',') + 1) AS remaining_prerequisites
        FROM courses_with_prerequisites
        WHERE remaining_prerequisites != ''
    ),

    courses_with_prerequisites_no_nulls AS (
        SELECT course_id,
            prerequisite
        FROM courses_with_prerequisites
        WHERE prerequisite != ''
        AND prerequisite IS NOT NULL
        GROUP BY course_id,
            prerequisite
    ),

    courses_with_prerequisite_counts AS (
        SELECT base.course_id,
            base.prerequisite,
            counts.total_prerequisites
        FROM courses_with_prerequisites_no_nulls base
        INNER JOIN
        (
            SELECT course_id,
                COUNT(prerequisite) AS total_prerequisites
            FROM courses_with_prerequisites_no_nulls
            GROUP BY course_id
        ) counts
        ON base.course_id = counts.course_id
    ),

    enrollments_base AS (
        SELECT participant_id,
            participant_name,
            course_id,
            course_date
        FROM enrollments
    ),

    enrollments_courses AS (
        SELECT E.participant_id,
            E.participant_name,
            E.course_id,
            E.course_date,
            C.total_prerequisites,
            C.prerequisite AS course_id_prerequisite
        FROM enrollments_base E
        LEFT JOIN courses_with_prerequisite_counts C
        ON E.course_id = C.course_id
    ),

    enrollments_with_earliest_enrollment AS (
        SELECT participant_id,
            course_id,
            course_date
        FROM
        (
            SELECT participant_id,
                course_id,
                course_date,
                ROW_NUMBER() OVER(PARTITION BY participant_id, course_id ORDER BY course_date) AS enrollment_order
            FROM enrollments_base
        ) AS enrollments_with_rn
        WHERE enrollment_order = 1
    ),

    enrollments_courses_prerequisite_date AS (
        SELECT EnrollmentsCourses.*,
            EarliestCompletion.course_date as course_date_prerequisite,
            CASE 
                WHEN EarliestCompletion.course_date IS NULL
                OR 

            END AS prerequisite_completion_status
        FROM enrollments_courses AS EnrollmentsCourses
        LEFT JOIN enrollments_with_earliest_enrollment AS EarliestCompletion
        ON EnrollmentsCourses.participant_id = EarliestCompletion.participant_id
        AND EnrollmentsCourses.course_id_prerequisite = EarliestCompletion.course_id
    )

    SELECT *
    FROM enrollments_courses_prerequisite_date
    ;
"""
results = execute_db_select(conn, reporting_query)
print(results)
conn.close()

   participant_id participant_name  course_id course_date  \
0            P001        Alice Tan          1  2024-01-15   
1            P002          Bob Lee          1  2024-01-15   
2            P007        Grace Koh          1  2024-01-20   
3            P009       Irene Chua          1  2024-01-25   
4            P003       Charlie Ng          2  2024-02-01   
5            P001        Alice Tan          2  2024-02-01   
6            P004        Diana Lim          1  2024-02-10   
7            P007        Grace Koh          2  2024-02-15   
8            P009       Irene Chua          2  2024-02-20   
9            P002          Bob Lee          3  2024-03-05   
10           P002          Bob Lee          3  2024-03-05   
11           P005       Edward Goh          4  2024-03-15   
12           P003       Charlie Ng          1  2024-03-20   
13           P008        Henry Teo          3  2024-03-25   
14           P008        Henry Teo          3  2024-03-25   
15           P007       